# Chapter 9 - HyperParameter Tuning with Cross-Validation

## Preparation

In [ ]:
import os
import sys
sys.path.append(os.path.abspath(os.path.join('..')))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.ensemble import RandomForestClassifier
from sklearn.datasets import make_classification
from sklearn.decomposition import PCA
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV
from sklearn.svm import SVC

from utils.sampling_bars import dollar_bar
from utils.cv import PurgedKFold
from utils.log_uniform import log_uniform

%matplotlib inline
plt.style.use('ggplot')
plt.rcParams['figure.figsize'] = 16,6

data_path = '../data/processed/clean_IVE_tickbidask.parq'
df = pd.read_parquet(data_path)

dollar_bars = dollar_bar(df)

## 1. Using the function getTestData from Chapter 8, form a synthetic dataset of  10,000 observations with 10 features, where 5 are informative and 5 are noise.

In [ ]:
def get_test_data(n_features=40, n_informative=10, n_redundant=10, n_samples=10000):
    # generate a random dataset for a classification problem    
    trains_df, labels_df = make_classification(
        n_samples=n_samples, 
        n_features=n_features, 
        n_informative=n_informative, 
        n_redundant=n_redundant, 
        random_state=0, 
        shuffle=False
    )
    # Create a DatetimeIndex using start and periods (not end)
    indices = pd.date_range(
        start=pd.Timestamp.today(), 
        periods=n_samples, 
        freq=pd.tseries.offsets.Minute()
    )
    trains_df = pd.DataFrame(trains_df, index=indices)
    labels_df = pd.Series(labels_df, index=indices).to_frame('bin')
    
    columns = ['I_%s' % i for i in range(n_informative)] + ['R_%s' % i for i in range(n_redundant)]
    columns += ['N_%s' % i for i in range(n_features - len(columns))]
    trains_df.columns = columns

    labels_df['w'] = 1.0 / labels_df.shape[0]
    labels_df['t1'] = pd.Series(labels_df.index, index=labels_df.index)
    return trains_df, labels_df

In [ ]:
trains_df, labels_df = get_test_data(n_features=10, n_informative=5, n_redundant=0, n_samples=10000)

### a. Use GridSearchCV on 10-fold CV to find the C, gamma optimal  hyperparameters on a SVC with RBF kernel, where  param_grid = {'C':[1E-2,1E-1,1,10,100],'gamma':[1E-2,1E1,1,10,100]} and the scoring function is neg_log_loss.

In [ ]:
purged_cv = PurgedKFold(n_splits=10, t1=labels_df.index.to_series())

grid_search = GridSearchCV(
    estimator=SVC(kernel='rbf', probability=True),
    param_grid={'C':[1e-2,1e-1,1,10,100],'gamma':[1e-2,1e1,1,10,100]},
    scoring='neg_log_loss',
    cv=purged_cv,
    n_jobs=-1, 
    iid=False, 
    return_train_score=True
)

grid_search.fit(trains_df, labels_df)

In [ ]:
best_params_gs = grid_search.best_params_
best_score_gs = grid_search.best_score_

grid_search_results = pd.DataFrame(grid_search.cv_results_)

print(f"Best parameters of grid search: {best_params_gs}")
print(f"Best score of grid search: {best_score_gs}")

### b. How many nodes are there in the grid?

In [ ]:
print(f"Number of nodes in the grid: {len(grid_search.param_grid['C']) * len(grid_search.param_grid['gamma'])}")

### c. How many fits did it take to find the optimal solution?

In [ ]:
print(f"Number of fits: {grid_search_results.shape[0]}")

### d. How long did it take to find this solution?

In [ ]:
print(f"It took {grid_search_results['mean_fit_time'].sum():.0f} seconds for grid search.")

### e. How can you access the optimal result?

In [ ]:
best_estimator = grid_search.best_estimator_

best_estimator

### f. What is the CV score of the optimal parameter combination?

In [ ]:
print(f"CV score of the optimal parameter combination: {grid_search.best_score_}")

### g. How can you pass sample weights to the SVC?

In [ ]:
sample_weights = labels_df['w']

best_estimator.fit(trains_df, labels_df, sample_weight=sample_weights)

best_estimator.score(trains_df, labels_df, sample_weight=sample_weights)

## 2. Using the same dataset from exercise 1,

### a. Use RandomizedSearchCV on 10-fold CV to find the C, gamma optimal  hyperparameters on an SVC with RBF kernel, where  param_distributions = {‘C’:logUniform(a = 1E-2,b =  1E2),‘gamma’:logUniform(a = 1E-2,b = 1E2)},n_iter = 25 and neg_log_loss is the scoring function.

In [ ]:
purged_cv = PurgedKFold(n_splits=10, t1=labels_df.index.to_series())

param_distributions = {'C': log_uniform(a=1e-2, b=1e2), 'gamma': log_uniform(a=1e-2, b=1e2)}

n_iter = 25
random_search = RandomizedSearchCV(
    estimator=SVC(kernel='rbf', 
    probability=True), 
    param_distributions=param_distributions, 
    scoring='neg_log_loss',
    cv=purged_cv, n_jobs=-1, 
    iid=False, 
    n_iter=n_iter, 
    return_train_score=True
)

random_search = random_search.fit(X=trains_df, y=labels_df['bin'])
random_search_results = pd.DataFrame(random_search.cv_results_)

### b. How long did it take to find this solution?

In [ ]:
print(f"It took {random_search_results['mean_fit_time'].sum():.0f} seconds for random search.")

### c. Is the optimal parameter combination similar to the one found in exercise  1?

In [ ]:
best_params_rs = random_search.best_params_
best_score_rs = random_search.best_score_

print(f"Best parameters of random search: {best_params_rs}")
print(f"Best score of random search: {best_score_rs}")

print()

print(f"Best parameters of grid search: {best_params_gs}")
print(f"Best score of grid search: {best_score_gs}")


### d. What is the CV score of the optimal parameter combination? How does it  compare to the CV score from exercise 1?

In [ ]:
print(f"CV score of the optimal parameter combination of random search: {random_search.best_score_}")
print(f"CV score of the optimal parameter combination of grid search: {grid_search.best_score_}")